In [1]:
import os
# os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.chdir('../')
import warnings
import scanpy as sc
import pandas as pd
import numpy as np

In [2]:
# the data path
main_dir = '/home/wergillius/Project/diffuse_differentiate/'
data_dir = '/home/wergillius/Project/diffuse_differentiate/data/Fetal_reference/'

In [ ]:
fetal_h5 = os.path.join(data_dir, 'GSE156793_protein_coding.h5ad')
adata_fetal = sc.read_h5ad(fetal_h5)
adata_fetal.var.reset_index(inplace=True)

NameError: name 'os' is not defined

In [ ]:
flatten_exp = adata_fetal.X.mean(axis=0).flatten()
adata_fetal.var['Mean_expression'] = flatten_exp.T

adata_fetal.obs.index = adata_fetal.obs['sample']
protein_coding_depth = adata_fetal.X.sum(axis=1)
adata_fetal.obs['protein_coding_depth'] = protein_coding_depth

# adata_fetal.write_h5ad(fetal_h5)

In [84]:
adata_fetal.obs['Matched_MCA_cell_name'].nunique()

189

In [85]:
adata_fetal.obs['MCA_beta'].nunique()

605

In [86]:
adata_fetal.obs['Matched_BCA_cell_name'].nunique()

43

In [4]:
fetal_h5 = os.path.join(data_dir, 'GSE156793_protein_coding.h5ad')
adata_fetal = sc.read_h5ad(fetal_h5)
sc.pp.filter_cells(adata_fetal, min_counts=100)
sc.pp.filter_genes(adata_fetal, min_cells=15)

/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [88]:
adata_fetal.obs['MCA_beta'].nunique()

605

In [89]:
adata_fetal.obs['Matched_MCA_cell_name'].nunique()

189

In [90]:
adata_fetal

AnnData object with n_obs × n_vars = 4062980 × 19268
    obs: 'All_reads', 'Assay', 'Batch', 'Development_day', 'Exon_reads', 'Experiment_batch', 'Fetus_id', 'Intron_reads', 'Main_cluster_name', 'Main_cluster_umap_1', 'Main_cluster_umap_2', 'Organ', 'Organ_cell_lineage', 'RT_group', 'Sex', 'Size_Factor', 'batch', 'obs_names', 'sample', 'subcluster_umap_1', 'subcluster_umap_2', 'sub_cluster_id', 'sub_cluster_name', 'Matched_MCA_cell_name', 'MCA_beta', 'Matched_BCA_cell_name', 'BCA_beta', 'BCA_cluster_info', 'Global_umap_1', 'Global_umap_2', 'protein_coding_depth', 'n_counts'
    var: 'level_0', 'exon_intron', 'gene_id', 'gene_short_name', 'gene_type', 'index', 'var_names', 'Mean_expression', 'n_cells'

In [5]:
fetal_hvg = adata_fetal.copy()
sc.pp.normalize_total(fetal_hvg, target_sum=1e4)
sc.pp.log1p(fetal_hvg)

In [6]:
sc.pp.highly_variable_genes(fetal_hvg, n_top_genes=5000)
# This saves the original set of genes 

# fetal_hvg = adata_fetal[:,adata_fetal.var.highly_variable]
# sc.pp.scale(fetal_hvg, max_value=10)

# read diff TF atlas 

In [7]:
adata_diff = sc.read_h5ad(f"{main_dir}/data/TFAtlas/GSE217460_210322_TFAtlas_differentiated.h5ad")

In [8]:
overlapped_genes = np.intersect1d(adata_fetal.var['gene_short_name'].values, adata_diff.var.index)
print('common genes ', len(overlapped_genes))

overlapped_hvg = np.intersect1d(fetal_hvg.var.query("`highly_variable` == True")['gene_short_name'].values, adata_diff.var.index)
print('common variable genes ', len(overlapped_hvg))

common genes  3135
common variable genes  1159


In [9]:
fvar_df = fetal_hvg.var.copy()

In [10]:
fvar_df = fvar_df.query("`gene_short_name` in @overlapped_hvg").sort_values('Mean_expression', ascending=False)
fvar_df = fvar_df.drop_duplicates(['gene_short_name'], keep='first')

In [11]:
com_hvg_index = fvar_df['level_0'].values

In [12]:
fetal_hvg = fetal_hvg[:, com_hvg_index].copy()
fetal_hvg.var.index = fvar_df['gene_short_name'].values

In [13]:
fetal_hvg = fetal_hvg[:,overlapped_hvg]

In [14]:
adata_diff  = adata_diff[:,overlapped_hvg].copy()

In [15]:
adata_Int = fetal_hvg.concatenate(adata_diff, batch_categories=['Fetal_atlas','TF_atlas'])

/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/anndata/_core/merge.py:942: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(
/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/anndata/_core/anndata.py:1785: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  [AnnData(sparse.csr_matrix(a.shape), obs=a.obs) for a in all_adatas],
/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/anndata/_core/anndata.py:1785: FutureWarning: X.dtype being converted to np.float32 from flo

In [16]:
sc.tl.pca(adata_Int, svd_solver='arpack')

In [17]:
adata_Int.write_h5ad(f"{main_dir}/data/TFAtlas/Be4_IntFetal_hvg.h5ad")

In [18]:
os.system("beep -f 555 -l 460")

sh: beep: command not found


32512

In [120]:
sc.pp.neighbors(adata_Int, n_neighbors=15, n_pcs=50)
sc.tl.umap(adata_Int)
sc.tl.leiden(adata_Int, resolution=0.5)

/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/umap/spectral.py:342: UserWarning: Exited at iteration 20 with accuracies 
[0.01424761 0.01719226 0.01752068]
not reaching the requested tolerance 1e-08.
Use iteration 21 instead with accuracy 
0.0163201864751155.

  eigenvalues, eigenvectors = scipy.sparse.linalg.lobpcg(
/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/umap/spectral.py:342: UserWarning: Exited postprocessing with accuracies 
[0.01424761 0.01719226 0.01752068]
not reaching the requested tolerance 1e-08.
  eigenvalues, eigenvectors = scipy.sparse.linalg.lobpcg(


: 

In [ ]:
sce.pp.harmony_integrate(adata_Int, 'batch', max_iter_harmony = 30, max_iter_kmeans = 50)

In [ ]:
adata_Int['max_iter_harmony = 30, max_iter_kmeans = 50']

# TF atlas

In [8]:
adata_TF = sc.read_h5ad(
    f"{main_dir}/data/TFAtlas/GSE217460_210322_TFAtlas.h5ad"
)

# adata_subTF = sc.read_h5ad(f"{main_dir}/data/TFAtlas/GSE217460_210322_TFAtlas_subsample.h5ad")
# adata_subTF.obs.reset_index(inplace=True)
# adata_subTF.obs.rename({"index":"cellbarcode"},axis=1,inplace=True)

In [19]:
adata_TF

AnnData object with n_obs × n_vars = 1145823 × 37528
    obs: 'TF', 'batch', 'louvain', 'n_counts', 'n_genes', 'percent_mito'

In [52]:
adata_TF.layers

Layers with keys: 

In [20]:
all_token = adata_TF.obs.TF.unique().tolist()

In [48]:
with open(f'{main_dir}/data/TFAtlas/tokens.txt', 'w') as f:
    for token in all_token:
        f.write('%s\n' %token)
    f.close()

In [32]:
token_dict = {token:int(token.split("-")[0].replace("TFORF","")) for token in all_token}

In [34]:
idx_token_dict = {i:t for t, i in token_dict.items()}

In [53]:
max(idx_token_dict.keys())

3550

In [ ]:
all_token.remove("Control")
all_token = ['Control'] + all_token

unique_token_dict = {k:i for i,k in enumerate(all_token)}

group0.uns['unique_token_dict'] = unique_token_dict

In [18]:
adata_TF.X.mean(axis=0)

array([0.02710287, 0.00204172, 0.00711865, ..., 0.13081132, 0.18087532,
       0.6210052 ], dtype=float32)

In [16]:
adata_TF.obs['n_counts']

R1.01,R2.01,R3.01,P1.22-0-0    2186.0
R1.01,R2.01,R3.02,P1.22-0-0    2605.0
R1.01,R2.01,R3.05,P1.38-0-0    3078.0
R1.01,R2.01,R3.09,P1.46-0-0    3164.0
R1.01,R2.01,R3.10,P1.22-0-0    1590.0
                                ...  
R1.96,R2.96,R3.45,P1.62-3-1    4225.0
R1.96,R2.96,R3.58,P1.62-3-1    2737.0
R1.96,R2.96,R3.79,P1.62-3-1    3581.0
R1.96,R2.96,R3.85,P1.62-3-1    4581.0
R1.96,R2.96,R3.87,P1.62-3-1    7833.0
Name: n_counts, Length: 1145823, dtype: float32

In [7]:
# adata_subTF.obs.index = adata_subTF.obs

In [8]:
import scanpy.external as sce

In [9]:
adata_subTF

AnnData object with n_obs × n_vars = 671453 × 3517
    obs: 'TF', 'batch', 'louvain', 'n_counts', 'n_genes', 'percent_mito', 'dpt_pseudotime', 'v_pseudotime'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'diffmap_evals', 'draw_graph', 'hvg', 'iroot', 'louvain', 'louvain_colors', 'neighbors', 'pca', 'rank_genes_groups', 'umap'
    obsm: 'NMF', 'X_diffmap', 'X_draw_graph_fa', 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [10]:
adata_subTF.obs.index

Index(['R1.07,R2.80,R3.02,P1.38-0-0', 'R1.08,R2.52,R3.41,P1.30-0-0',
       'R1.09,R2.21,R3.87,P1.30-0-0', 'R1.10,R2.84,R3.12,P1.46-0-0',
       'R1.11,R2.84,R3.45,P1.46-0-0', 'R1.12,R2.79,R3.24,P1.30-0-0',
       'R1.14,R2.34,R3.80,P1.38-0-0', 'R1.17,R2.02,R3.11,P1.22-0-0',
       'R1.17,R2.65,R3.78,P1.38-0-0', 'R1.19,R2.02,R3.86,P1.46-0-0',
       ...
       'R1.95,R2.49,R3.84,P1.38-2-1', 'R1.06,R2.27,R3.89,P1.62-3-1',
       'R1.09,R2.16,R3.68,P1.62-3-1', 'R1.19,R2.38,R3.67,P1.62-3-1',
       'R1.20,R2.72,R3.85,P1.62-3-1', 'R1.29,R2.64,R3.18,P1.62-3-1',
       'R1.36,R2.31,R3.09,P1.62-3-1', 'R1.52,R2.33,R3.67,P1.62-3-1',
       'R1.87,R2.35,R3.24,P1.62-3-1', 'R1.92,R2.91,R3.74,P1.62-3-1'],
      dtype='object', length=671453)

In [12]:
adata_subTF.var.index

Index(['AAK1', 'AARS', 'AATF', 'ABCA1', 'ABCA4', 'ABCB11', 'ABCB5', 'ABCC1',
       'ABCE1', 'ABCF1',
       ...
       'ZNF91', 'ZNHIT3', 'ZNRF1', 'ZNRF2', 'ZNRF3', 'ZRANB2-AS2', 'ZRANB3',
       'ZSWIM5', 'ZSWIM6', 'ZZZ3'],
      dtype='object', length=3517)

In [26]:
overlap_genes =  np.intersect1d(adata_fetal.var['gene_short_name'].values,  adata_subTF.var.index)

In [29]:
dup_var = adata_fetal.var[adata_fetal.var['gene_short_name'].isin(overlap_genes)]
dup_var = dup_var.sort_values('Mean_expression', ascending=False).drop_duplicates("gene_short_name", keep='first')
overlap_index = dup_var.index

In [37]:
og_subTF = adata_subTF[:,overlap_genes].copy()

In [38]:
og_fetal = adata_fetal[:,overlap_index].copy()

/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/anndata/_core/anndata.py:121: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [46]:
og_fetal.var.index = og_fetal.var['gene_short_name']

In [48]:
adata_int = og_fetal.concatenate(og_subTF, batch_categories=['Fetal_atlas','TF_atlas'])

/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/anndata/_core/merge.py:942: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(
/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/anndata/_core/anndata.py:1785: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  [AnnData(sparse.csr_matrix(a.shape), obs=a.obs) for a in all_adatas],
/home/wergillius/.conda/envs/GenDiff/lib/python3.8/site-packages/anndata/_core/anndata.py:1785: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  [AnnData(sparse.csr_matrix(a.shape),

In [50]:
sc.pp.filter_cells(adata_int, min_counts=100)
sc.pp.filter_genes(adata_int, min_cells=5)

In [51]:
adata_int.var['mt'] = adata_int.var_names.str.startswith('MT-')

In [53]:
adata_int.var['mt'].sum()

0

In [59]:
og_subTF.X.max(axis=0)

array([4.3417835, 4.327071 , 4.808628 , ..., 8.174618 , 4.7162957,
       4.2594876], dtype=float32)

In [60]:
og_subTF.X.min(axis=0)

array([-1.4282931, -1.5349555, -1.2466139, ..., -0.7380165, -1.3647482,
       -1.541671 ], dtype=float32)

In [56]:
adata_int.X[-10:,:10].todense()

matrix([[-0.9298153 , -0.3726727 , -0.8701277 , -0.33791777, -0.28268194,
         -0.15569629,  0.9252982 ,  0.8896374 , -0.41431212, -0.40286416],
        [-1.0901843 ,  1.6888502 ,  0.2802957 , -0.41028827, -0.3564375 ,
         -0.1783742 , -0.99545455, -0.9979646 , -0.49619853, -0.4923843 ],
        [-0.90041304, -0.355305  , -0.8429505 , -0.32464927, -0.2691595 ,
         -0.15153849, -0.7951402 , -0.82852674,  2.384861  , -0.38645142],
        [ 1.0323321 ,  1.5701232 , -1.0482336 , -0.42487323, -0.37130156,
         -0.1829445 , -1.0295695 , -1.0268211 , -0.5127013 ,  1.422198  ],
        [ 0.6361883 , -0.38496163, -0.88935745, -0.3473062 , -0.29225004,
         -0.15863822,  0.82889277,  0.79691696, -0.42493504, -0.41447735],
        [ 1.9307058 , -0.36002102, -0.85033023, -0.32825223, -0.27283138,
         -0.1526675 , -0.8035677 ,  0.99357915, -0.40337566, -0.39090812],
        [-1.0697639 , -0.45533952,  0.9670396 , -0.40107307, -0.34704593,
         -0.17548653, -0.9738996

In [ ]:
sce.pp.harmony_integrate(adata_int, 'batch')